# 2.1 每字的一排可調數字，為什麼能幫忙猜下一字？


上一章每字直接查出一整排候選分數。現在想讓幾個字共同參與預測，先替每字準備一小排可調數字，後面再用共用配方將這些數字變成候選分數。這排中間數字叫**特徵向量**：向量是一排數，特徵是供後續計算使用的線索，不必每格都有「動物」這類人類名稱。

為了手動填一張五列的特徵表，本節使用新的列順序：0 是句號、1 是貓、2 是看、3 是狗、4 是逗號。ID 仍只是選列約定；接下來的文字與 ID 都按這張表對應。每字先存三數；看到「貓看貓」，就按 `[1,2,1]` 查三次。

| ID／字 | 本節手動指定的三個特徵 |
| --- | --- |
| 0／。 | `[0,0,0]` |
| 1／貓 | `[1,0,0]` |
| 2／看 | `[0,1,0]` |
| 3／狗 | `[0,0,1]` |
| 4／， | `[1,1,1]` |

為什麼不直接用 ID？ID 只選列，數字 1 或 2 本身不代表字義；另外存一排參數，才讓訓練能改「這個字提供哪些數字」，又不改編碼約定。這張表與查出的表示叫 **embedding**，即嵌入表示。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch import nn

embedding = nn.Embedding(5, 3)
with torch.no_grad():
    embedding.weight.copy_(
        torch.tensor(
            [
                [0.0, 0.0, 0.0],
                [1.0, 0.0, 0.0],
                [0.0, 1.0, 0.0],
                [0.0, 0.0, 1.0],
                [1.0, 1.0, 1.0],
            ]
        )
    )
ids = torch.tensor([[1, 2, 1]])
x = embedding(ids)
print(x.shape)
print(x)
assert torch.equal(x[0, 0], x[0, 2])

輸出是 `[1,3,3]`：一筆、三位置、每位置三特徵；三排依序是 `[1,0,0]`、`[0,1,0]`、`[1,0,0]`。兩次貓使用同一列，所以 `torch.equal` 的檢查成立。

查表後還要做什麼？先看一個額外的手算配方，並未放進上面的程式：假設候選「狗」的分數為 `2×第一格+第三格`。貓的 `[1,0,0]` 會算出 2，看的 `[0,1,0]` 會算出 0。同一配方遇到不同字，就因查出的特徵不同而給不同分數。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-02-embedding-purpose.svg")))

圖的上半是查表，下半接到剛才的候選配方。假設稍後把貓列第一格由 1 調成 1.5，狗分數會由 2 變 3；調那個配方的乘數也會改分數。模型便能同時調字表與後續配方，讓正確下一字取得更高機率。這裡只是說明數字如何影響預測，沒有用這個假設證明訓練已成功。

真正訓練時，答案代價的梯度沿運算關係傳回用到的表格列，再由更新步驟改表；只建立 `nn.Embedding` 不會自動學會關係。未手動填表時，它先從亂數開始。

練習將第二格 ID 2 改成 1，三個位置便都查出 `[1,0,0]`，形狀保持不變。同一字起始向量相同；加入鄰字與位置後，各位置的中間表示才可能不同。

<details>
<summary>補充與重做</summary>

查表工具見 [1.6](https://birdhackor.github.io/tiny-perceptron-vlm/1.6.html)，梯度傳回與更新見 [1.10](https://birdhackor.github.io/tiny-perceptron-vlm/1.10.html)、[1.12](https://birdhackor.github.io/tiny-perceptron-vlm/1.12.html)，三個軸見 [W.3](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.3)。V 個字、每字 D 格的表有 V×D 個參數，D 不必等於候選數 V。上面的候選配方是手算示意，並非程式已建立的輸出層。

</details>
